# 02 - Preprocessing & Feature Engineering

**Objectif** : Préparer les données pour l'entraînement des modèles ML.

**Étapes** :
1. Suppression des variables redondantes
2. Encodage des variables catégorielles
3. Feature engineering
4. Transformation de la cible (log)
5. Split train/test
6. Sauvegarde des données préparées

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Chargement et nettoyage

In [2]:
df = pd.read_csv('../data/raw/CMR DS10 ML training set.csv')
print(f"Shape initiale: {df.shape}")

# Suppression de co2_ppm (corrélation 0.97 avec année)
# Suppression de production_t (= rendement × superficie, pas une feature)
df = df.drop(columns=['co2_ppm', 'production_t'])
print(f"Shape après suppression co2/production: {df.shape}")

Shape initiale: (6000, 24)
Shape après suppression co2/production: (6000, 22)


## 2. Feature Engineering

In [3]:
# Ratio NPK - équilibre de la fertilisation
df['NK_ratio'] = df['N_kgha'] / (df['K_kgha'] + 1)
df['NP_ratio'] = df['N_kgha'] / (df['P_kgha'] + 1)
df['NPK_total'] = df['N_kgha'] + df['P_kgha'] + df['K_kgha']

# Indice d'aridité (ETP / précipitations)
df['indice_aridite'] = (df['etp_mmd'] * 365) / (df['precipitations_mm'] + 1)

# Bilan hydrique simplifié
df['bilan_hydrique'] = df['precipitations_mm'] - (df['etp_mmd'] * 365)

# Interaction température × humidité
df['temp_x_humidite'] = df['temperature_C'] * df['humidite_pct'] / 100

print(f"Shape après feature engineering: {df.shape}")
print(f"\nNouvelles features: NK_ratio, NP_ratio, NPK_total, indice_aridite, bilan_hydrique, temp_x_humidite")

Shape après feature engineering: (6000, 28)

Nouvelles features: NK_ratio, NP_ratio, NPK_total, indice_aridite, bilan_hydrique, temp_x_humidite


## 3. Encodage des variables catégorielles

In [4]:
cat_cols = ['region', 'culture', 'saison', 'type_sol', 'irrigation', 'pratique_agricole']

# Label Encoding pour chaque variable catégorielle
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"{col}: {len(le.classes_)} catégories → {list(le.classes_)[:5]}...")

# Sauvegarder le mapping pour le déploiement
encoding_map = {col: dict(zip(le.classes_, le.transform(le.classes_)))
                for col, le in label_encoders.items()}

print(f"\nEncodings sauvegardés pour {len(encoding_map)} variables")

region: 10 catégories → ['Adamaoua', 'Centre', 'Est', 'Extrême-Nord', 'Littoral']...
culture: 24 catégories → ['ananas', 'arachide', 'bananier', 'cacao', 'cacaotier']...
saison: 4 catégories → ['grande_saison_pluies', 'petite_saison_pluies', 'petite_saison_sèche', 'saison_sèche']...
type_sol: 16 catégories → ['alluvial', 'andosolique', 'brun_eutrophe', 'brun_subaride', 'ferrallitique']...
irrigation: 5 catégories → ['aspersion', 'goutte-à-goutte', 'gravitaire', 'non_irriguée', 'pluviale']...
pratique_agricole: 5 catégories → ['améliorée', 'biologique', 'conventionnelle', 'intégrée', 'traditionnelle']...

Encodings sauvegardés pour 6 variables


## 4. Préparation des features et de la cible

In [5]:
# Features à utiliser pour le modèle
feature_cols = [
    # Numériques originales
    'annee', 'temperature_C', 'precipitations_mm', 'humidite_pct',
    'rayonnement_MJm2', 'etp_mmd', 'usage_terres_pct', 'stress_ecologique_idx',
    'ph_sol', 'matiere_organique_pct', 'N_kgha', 'P_kgha', 'K_kgha',
    'altitude_m', 'superficie_ha',
    # Features engineered
    'NK_ratio', 'NP_ratio', 'NPK_total', 'indice_aridite',
    'bilan_hydrique', 'temp_x_humidite',
    # Catégorielles encodées
    'region_encoded', 'culture_encoded', 'saison_encoded',
    'type_sol_encoded', 'irrigation_encoded', 'pratique_agricole_encoded'
]

X = df[feature_cols]
y = df['rendement_tha']

# Transformation log de la cible (pour réduire le skew)
y_log = np.log1p(y)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nSkew avant log: {y.skew():.2f}")
print(f"Skew après log: {y_log.skew():.2f}")

X shape: (6000, 27)
y shape: (6000,)

Skew avant log: 1.49
Skew après log: 0.49


## 5. Split Train / Test

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} échantillons")
print(f"Test:  {X_test.shape[0]} échantillons")
print(f"\nDistribution cible (log) - Train: mean={y_train.mean():.2f}, std={y_train.std():.2f}")
print(f"Distribution cible (log) - Test:  mean={y_test.mean():.2f}, std={y_test.std():.2f}")

Train: 4800 échantillons
Test:  1200 échantillons

Distribution cible (log) - Train: mean=1.40, std=0.93
Distribution cible (log) - Test:  mean=1.46, std=0.96


## 6. Normalisation (pour CNN-LSTM)

In [7]:
# Normalisation StandardScaler (nécessaire pour le réseau de neurones, pas pour XGBoost/RF)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
    index=X_test.index
)

print("Normalisation effectuée (mean=0, std=1)")
print(f"\nVérification X_train_scaled:")
print(f"  Mean: {X_train_scaled.mean().mean():.6f}")
print(f"  Std:  {X_train_scaled.std().mean():.4f}")

Normalisation effectuée (mean=0, std=1)

Vérification X_train_scaled:
  Mean: -0.000000
  Std:  1.0001


## 7. Sauvegarde

In [8]:
import joblib

# Données pour XGBoost et Random Forest (pas normalisées)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Données normalisées pour CNN-LSTM
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)

# Sauvegarder les objets de preprocessing
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(label_encoders, '../models/label_encoders.joblib')
joblib.dump(feature_cols, '../models/feature_cols.joblib')

# Sauvegarder le dataset complet avec features engineered
df.to_csv('../data/processed/dataset_preprocessed.csv', index=False)

print("Fichiers sauvegardés:")
print("  data/processed/X_train.csv")
print("  data/processed/X_test.csv")
print("  data/processed/y_train.csv")
print("  data/processed/y_test.csv")
print("  data/processed/X_train_scaled.csv")
print("  data/processed/X_test_scaled.csv")
print("  models/scaler.joblib")
print("  models/label_encoders.joblib")
print("  models/feature_cols.joblib")

Fichiers sauvegardés:
  data/processed/X_train.csv
  data/processed/X_test.csv
  data/processed/y_train.csv
  data/processed/y_test.csv
  data/processed/X_train_scaled.csv
  data/processed/X_test_scaled.csv
  models/scaler.joblib
  models/label_encoders.joblib
  models/feature_cols.joblib


## Résumé du preprocessing

| Étape | Action |
|-------|--------|
| Suppression | `co2_ppm` (redondant), `production_t` (dérivée) |
| Feature Engineering | 6 nouvelles features (ratios NPK, indices climatiques) |
| Encodage | Label Encoding sur 6 variables catégorielles |
| Transformation cible | log(1 + rendement) pour réduire le skew |
| Split | 80% train (4800) / 20% test (1200) |
| Normalisation | StandardScaler pour CNN-LSTM |